In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, IntegerType,
    DateType, TimestampType, DoubleType, BooleanType
)

GOLD_BASE = "abfss://deltalake@mkazureproject1.dfs.core.windows.net/gold"

def create_gold_table(table_name, schema, partition_col=None):
    builder = (
        DeltaTable.createIfNotExists(spark)
            .tableName(f"spotify_catalog.gold.{table_name}")
            .addColumns(schema)
            .property("delta.autoOptimize.optimizeWrite",   "true")
            .property("delta.autoOptimize.autoCompact",     "true")
            .property("delta.enableChangeDataFeed",         "true")
            .property("delta.logRetentionDuration",         "interval 90 days")
            .location(f"{GOLD_BASE}/{table_name}/data")
    )
    if partition_col:
        builder = builder.partitionedBy(partition_col)
    builder.execute()
    print(f"spotify_catalog.gold.{table_name} ready")

# ── Gold table schemas ────────────────────────────────────────────────────────

# KPI 1 — genre + subscription engagement
schema_genre_subscription = StructType([
    StructField("genre",                    StringType(),  True),
    StructField("subscription_type",        StringType(),  True),
    StructField("total_streams",            LongType(),    True),
    StructField("unique_users",             LongType(),    True),
    StructField("avg_listen_sec",           DoubleType(),  True),
    StructField("meaningful_plays",         LongType(),    True),
    StructField("abandoned_count",          LongType(),    True),
    StructField("meaningful_play_rate_pct", DoubleType(),  True),
    StructField("gold_ingest_timestamp",    TimestampType(), True),
])

# KPI 2 — top tracks
schema_top_tracks = StructType([
    StructField("track_id",            LongType(),    True),
    StructField("track_name",          StringType(),  True),
    StructField("artist_name",         StringType(),  True),
    StructField("genre",               StringType(),  True),
    StructField("total_streams",       LongType(),    True),
    StructField("unique_listeners",    LongType(),    True),
    StructField("total_listen_hrs",    DoubleType(),  True),
    StructField("avg_completion_pct",  DoubleType(),  True),
    StructField("skip_rate_pct",       DoubleType(),  True),
    StructField("gold_ingest_timestamp", TimestampType(), True),
])

# KPI 3 — device + subscription
schema_device_subscription = StructType([
    StructField("subscription_type",   StringType(),  True),
    StructField("device_type",         StringType(),  True),
    StructField("total_streams",       LongType(),    True),
    StructField("unique_users",        LongType(),    True),
    StructField("avg_listen_sec",      DoubleType(),  True),
    StructField("abandon_rate_pct",    DoubleType(),  True),
    StructField("gold_ingest_timestamp", TimestampType(), True),
])

# KPI 4 — monthly trend
schema_monthly_trend = StructType([
    StructField("year",                  IntegerType(), True),
    StructField("month",                 IntegerType(), True),
    StructField("month_name",            StringType(),  True),
    StructField("total_streams",         LongType(),    True),
    StructField("active_users",          LongType(),    True),
    StructField("total_listen_hrs",      DoubleType(),  True),
    StructField("avg_completion_pct",    DoubleType(),  True),
    StructField("streams_per_user",      DoubleType(),  True),
    StructField("gold_ingest_timestamp", TimestampType(), True),
])

# KPI 5 — user engagement scorecard
schema_user_scorecard = StructType([
    StructField("user_sk",             LongType(),    True),
    StructField("user_id",             LongType(),    True),
    StructField("user_name",           StringType(),  True),
    StructField("user_country",        StringType(),  True),
    StructField("subscription_type",   StringType(),  True),
    StructField("total_streams",       LongType(),    True),
    StructField("meaningful_plays",    LongType(),    True),
    StructField("abandoned",           LongType(),    True),
    StructField("engagement_score",    LongType(),    True),
    StructField("engagement_tier",     StringType(),  True),
    StructField("rank_in_tier",        IntegerType(), True),
    StructField("gold_ingest_timestamp", TimestampType(), True),
])

# ── create ────────────────────────────────────────────────────────────────────
create_gold_table("genre_subscription_kpi", schema_genre_subscription)
create_gold_table("top_tracks_kpi",         schema_top_tracks)
create_gold_table("device_subscription_kpi", schema_device_subscription)
create_gold_table("monthly_trend_kpi",      schema_monthly_trend,  partition_col="year")
create_gold_table("user_engagement_kpi",    schema_user_scorecard)